In [1]:
import json
import numpy as np
from pandas import read_csv
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
import joblib


In [2]:
#Variables generales
ruta_model = "../../Model/"
ruta_metrics = "../../Metrics/"
ruta_df = "../../Datasets/"

semilla = 111

## FUNCIÓN BASE

In [3]:
def train_naive_bayes(X, y, output_model_path = 'best_model.pkl', output_metrics_path = 'metrics.json'):
    # Define hyperparameters
    param_grid = {
        'var_smoothing': np.logspace(0, -9, num = 100)
    }
    
    # Initialize Naive Bayes
    nb = GaussianNB()
    
    # Define K-Fold Cross Validation
    kf = KFold(n_splits = 5
               , shuffle = True
               , random_state = 42
               )
    
    # Cross Validation
    grid_search = GridSearchCV(estimator=nb
                               , param_grid=param_grid
                               , cv=kf
                               , scoring='accuracy'
                               , n_jobs=-1
                               , verbose=2
                               )
    
    # Ajustar el modelo
    grid_search.fit(X, y)
    
    # mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guarde el mejor modelo en un archivo .pkl
    joblib.dump(best_model, output_model_path)
    
    # mejor modelo.
    y_pred = best_model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average = 'weighted'),
        'recall': recall_score(y, y_pred, average = 'weighted'),
        'f1_score': f1_score(y, y_pred, average = 'weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class = 'ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict = True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guarde las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)

    return metrics, best_model

## Entrenamiento

### df interpolation

* SMOTE

In [4]:
# carga de caracteristicas
df_IM_smote = read_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [5]:
metric_1, model_1 = train_naive_bayes(X = df_IM_smote.drop(columns='FLAG')
                                              , y = df_IM_smote['FLAG']
                                              , output_model_path = '{}G_IM_S.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}G_IM_S.json'.format(ruta_metrics))

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [6]:
metric_1

{'accuracy': 0.580511332284237,
 'precision': 0.6840135850224136,
 'recall': 0.580511332284237,
 'f1_score': 0.4602067501434191,
 'roc_auc': 0.5295765486234183,
 'confusion_matrix': [[26857, 327], [20199, 1548]],
 'classification_report': {'0': {'precision': 0.5707454947296838,
   'recall': 0.9879708652148322,
   'f1-score': 0.7235183189655173,
   'support': 27184},
  '1': {'precision': 0.8256,
   'recall': 0.07118223203200441,
   'f1-score': 0.13106426212852426,
   'support': 21747},
  'accuracy': 0.580511332284237,
  'macro avg': {'precision': 0.6981727473648419,
   'recall': 0.5295765486234183,
   'f1-score': 0.4272912905470208,
   'support': 48931},
  'weighted avg': {'precision': 0.6840135850224136,
   'recall': 0.580511332284237,
   'f1-score': 0.4602067501434191,
   'support': 48931}},
 'balanced_accuracy': 0.5295765486234183,
 'cohen_kappa': 0.06510223610153965,
 'matthews_corrcoef': 0.15311780952289833}

* ADASYN

In [7]:
# carga de caracteristicas
df_IM_adasyn = read_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [8]:
metric_2, model_2 = train_naive_bayes(X = df_IM_adasyn.drop(columns='FLAG')
                                              , y = df_IM_adasyn['FLAG']
                                              , output_model_path = '{}G_IM_A.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}G_IM_A.json'.format(ruta_metrics))

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [9]:
metric_2

{'accuracy': 0.575591985428051,
 'precision': 0.6564166160250631,
 'recall': 0.575591985428051,
 'f1_score': 0.45218348139432435,
 'roc_auc': 0.523602467292074,
 'confusion_matrix': [[26773, 411], [20326, 1351]],
 'classification_report': {'0': {'precision': 0.568440943544449,
   'recall': 0.9848808122424956,
   'f1-score': 0.7208378767685742,
   'support': 27184},
  '1': {'precision': 0.7667423382519863,
   'recall': 0.062324122341652445,
   'f1-score': 0.11527795554417851,
   'support': 21677},
  'accuracy': 0.575591985428051,
  'macro avg': {'precision': 0.6675916408982177,
   'recall': 0.523602467292074,
   'f1-score': 0.4180579161563764,
   'support': 48861},
  'weighted avg': {'precision': 0.6564166160250631,
   'recall': 0.575591985428051,
   'f1-score': 0.45218348139432435,
   'support': 48861}},
 'balanced_accuracy': 0.523602467292074,
 'cohen_kappa': 0.05204844959251853,
 'matthews_corrcoef': 0.12578674370099888}

### df linear regression

* SMOTE

In [10]:
# carga de caracteristicas
df_LR_smote = read_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [11]:
metric_3, model_3 = train_naive_bayes(X = df_LR_smote.drop(columns='FLAG')
                                              , y = df_LR_smote['FLAG']
                                              , output_model_path = '{}G_LR_S.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}G_LR_S.json'.format(ruta_metrics))

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [12]:
metric_3

{'accuracy': 0.5832294455457685,
 'precision': 0.6966775040034342,
 'recall': 0.5832294455457685,
 'f1_score': 0.46436961337457494,
 'roc_auc': 0.5324367059701198,
 'confusion_matrix': [[26900, 284], [20109, 1638]],
 'classification_report': {'0': {'precision': 0.5722308494118148,
   'recall': 0.9895526780459094,
   'f1-score': 0.7251357944819592,
   'support': 27184},
  '1': {'precision': 0.8522372528616025,
   'recall': 0.07532073389433025,
   'f1-score': 0.1384088892644387,
   'support': 21747},
  'accuracy': 0.5832294455457685,
  'macro avg': {'precision': 0.7122340511367087,
   'recall': 0.5324367059701198,
   'f1-score': 0.43177234187319896,
   'support': 48931},
  'weighted avg': {'precision': 0.6966775040034342,
   'recall': 0.5832294455457685,
   'f1-score': 0.46436961337457494,
   'support': 48931}},
 'balanced_accuracy': 0.5324367059701198,
 'cohen_kappa': 0.07138087455767206,
 'matthews_corrcoef': 0.165941839372339}

* ADASYN

In [13]:
# carga de caracteristicas
df_LR_adasyn = read_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [14]:
metric_4, model_4 = train_naive_bayes(X = df_LR_adasyn.drop(columns='FLAG')
                                              , y = df_LR_adasyn['FLAG']
                                              , output_model_path = '{}G_LR_A.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}G_LR_A.json'.format(ruta_metrics))

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [15]:
metric_4

{'accuracy': 0.5802403204272363,
 'precision': 0.6709039172446954,
 'recall': 0.5802403204272363,
 'f1_score': 0.4585965322010208,
 'roc_auc': 0.5265504824173008,
 'confusion_matrix': [[26817, 367], [20069, 1432]],
 'classification_report': {'0': {'precision': 0.5719617796357122,
   'recall': 0.9864994114184814,
   'f1-score': 0.7240988254353988,
   'support': 27184},
  '1': {'precision': 0.7959977765425236,
   'recall': 0.06660155341612017,
   'f1-score': 0.12291845493562231,
   'support': 21501},
  'accuracy': 0.5802403204272363,
  'macro avg': {'precision': 0.6839797780891179,
   'recall': 0.5265504824173008,
   'f1-score': 0.4235086401855106,
   'support': 48685},
  'weighted avg': {'precision': 0.6709039172446954,
   'recall': 0.5802403204272363,
   'f1-score': 0.4585965322010208,
   'support': 48685}},
 'balanced_accuracy': 0.5265504824173008,
 'cohen_kappa': 0.058725872994451866,
 'matthews_corrcoef': 0.1397819997466631}